# 02 · Clean the diagnose-me (MedDialog-EN) data and prepare it for patient-model fine-tuning

**SimClinic, CS425 Group 7.** Input: `data/raw/diagnose-me/en_medical_dialog.json`. This is the full English MedDialog scrape, with 257,469 rows from HealthCareMagic and iCliniq. Each row is `{id, Description, Patient, Doctor}`, which is one patient question and one doctor reply.

Outputs:
1. `data/processed/diagnose-me/clean/{train,dev,test}.jsonl`: cleaned dialogues that do not depend on any model, in the same schema as notebook 01.
2. `data/processed/diagnose-me/tokenized/<model_slug>/`: the tokenized `DatasetDict` (`input_ids`, `attention_mask`, `labels`). This is what the trainer reads.

Change `MODEL_NAME` in the config cell to target another Hugging Face model. Set `MAX_ROWS` to a small number for a quick test run.

**How this differs from the COVID set (notebook 01):**
- It is about 400× larger, so tokenization uses `Dataset.map(batched=True)` instead of Python loops.
- It has **no official splits**. The notebook makes a deterministic 90/5/5 split by hashing the patient text, which also keeps duplicates in the same split.
- The text is **mixed case**. Case is kept by default (`LOWERCASE = False`), and only ALL-CAPS messages are lowercased.
- There is more **personal information** (names, emails, phone numbers) and more references to attachments, which a chat patient cannot send.
- Doctor replies carry heavy site boilerplate and doctor signatures. iCliniq replies are often hollow, because the content was stripped and only the sign-off and a link remain.

## 0. Config

In [1]:
from pathlib import Path

# --- model -------------------------------------------------------------------
MODEL_NAME = "gpt2"          # try: "gpt2-medium", "distilgpt2", "EleutherAI/pythia-160m", "Qwen/Qwen2.5-0.5B"
MAX_LENGTH = 512             # clipped to the tokenizer's model_max_length
MAX_ROWS = None              # e.g. 5000 for a quick run; None = all 257k rows

# --- training target (same format as notebook 01) --------------------------------
TRAIN_ROLE = "patient"
ADD_DOCTOR_OPENER = True
DOCTOR_OPENER = "What brings you in today?"
TRIM_AFTER_LAST_TARGET = True  # single-turn data: the doctor reply is not in the training string
ROLE_TOKENS = {"doctor": "<|doctor|>", "patient": "<|patient|>"}
EXTRA_SPECIAL_TOKENS = ["<|disease|>", "<|symptoms|>", "<|persona|>", "<|diagnosis|>"]

# --- cleaning / filtering --------------------------------------------------------
LOWERCASE = False            # True to match the all-lowercase COVID set when mixing datasets
FIX_ALL_CAPS = True          # lowercase messages written entirely in capitals
MIN_WORDS, MAX_WORDS = 8, 300          # patient length window (raw p5 = 42, p99 = 296 words)
MAX_DIGIT_RATIO = 0.30       # drop lab-report dumps (share of tokens containing a digit)
DROP_ATTACHMENT_REFS = True  # "please find my reports attached": a chat patient cannot attach files
MIN_DOCTOR_WORDS = 3         # shorter doctor replies are blanked and flagged (doctor_usable = False)

# Removed from DOCTOR turns (case-insensitive, applied repeatedly until nothing changes).
DOCTOR_BOILERPLATE = [
    r"^(?:hi|hello|hey)(?: there)?[,.!]?\s*(?:dear\s*[,.]?\s*)?(?=thank|welcome|i have|i went|i (?:can )?understand|we understand)",
    r"^thanks? (?:you )?for (?:the|your) query to (?:my )?hcm(?: clinic)?\.?",
    r"\(attachment removed to protect patient identity\)",
    r"for (?:further|more) (?:information|doubts|queries)?\s*consult an? [^.>]*?online\s*-*>?",
    r"^hello and welcome to 'ask a doctor' service\.?",
    r"^i have reviewed your query and here is my advice\.?",
    r"^thanks? (?:you )?for (?:your|the) (?:question|query|concern)[^.]*\.",
    r"^(?:hi|hello)[,.]? thanks? (?:you )?for (?:using|writing to|posting on) [^.]*\.",
    r"^(?:hi|hello)?[,.]?\s*(?:dear)?\s*,?\s*(?:warm )?welcome to (?:hcm|healthcare ?magic(?:\.com)?)[,.]?",
    r"^dear\s*we understand your concerns?",
    r"^i (?:have )?(?:gone|went) through your (?:query|question|details|information)[^.]*\.",
    r"^(?:hi|hello)[,.]?\s*(?:dear)?\s*i have gone through your (?:query|question)[^.]*\.",
    r"^i (?:can )?understand your concerns?\.",
    r"hope (?:i have answered your query|this helps|that helps|this information helps)[^.]*\.?",
    r"let me know if i can (?:assist|help) you further\.?",
    r"wish(?:ing)? you (?:a very )?good health\.?\s*(?:thanks\.?)?",
    r"(?:best wishes|regards|take care|thanks|good luck)?[,.]?\s*(?:dr\.?|doctor)\s+[a-z][a-z .,&()-]{0,80}$",  # signature
    r"(?:best wishes|regards|take care)\W*$",
]

# --- paths ---------------------------------------------------------------------
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_FILE = ROOT / "data" / "raw" / "diagnose-me" / "en_medical_dialog.json"
OUT_DIR = ROOT / "data" / "processed" / "diagnose-me"
# Held-out sets that must not leak into this training data (skipped if missing).
EXCLUDE_JSONL = [ROOT / "data" / "processed" / "clean" / f"{s}.jsonl" for s in ("dev", "test")]
SPLIT_PCT = {"train": 90, "dev": 5, "test": 5}
SPLITS = list(SPLIT_PCT)
SEED = 42

## 1. Load and validate

In [2]:
import json, re, hashlib, unicodedata
import pandas as pd

pd.set_option("display.max_colwidth", 140)

df = pd.read_json(RAW_FILE)
assert set(df.columns) == {"id", "Description", "Doctor", "Patient"}, df.columns
assert df["id"].is_unique
if MAX_ROWS:
    df = df.sample(n=min(MAX_ROWS, len(df)), random_state=SEED).sort_values("id")
for col in ("Description", "Doctor", "Patient"):
    df[col] = df[col].fillna("").astype(str)

# iCliniq questions are titled "Q. ..." and its replies link to icliniq.com; the rest is HealthCareMagic.
df["source"] = df["Description"].str.startswith("Q.").map({True: "icliniq", False: "healthcaremagic"})

print(f"{len(df):,} rows")
print(df["source"].value_counts().to_string())
print("empty patient:", (df["Patient"].str.strip() == "").sum(), "| empty doctor:", (df["Doctor"].str.strip() == "").sum())

257,469 rows
source
healthcaremagic    226405
icliniq             31064
empty patient: 3 | empty doctor: 90


In [3]:
df.sample(3, random_state=1)

,id,Description,Doctor,Patient,source
154550,154550,What causes swelling in ankle?,"Hello, As you get hot spells than check the uric acid along with the x-ray. Because during an increase in the uric acid levels there wi...",the top of my foot hurts a lot when I press the center section of the bottom and cannot step too hard. It s hurts where it connects my a...,healthcaremagic
13758,13758,Q. Why do periods get delayed after first time sex?,"Hello. As the serum beta hCG levels are less than 3 mIU/mL, the pregnancy is definitely ruled out. The delay in the menses is mostly due...","Hi doctor, I had sex for the first time. After that, I did not get my periods yet. I went to a gynecologist. She had taken my blood samp...",icliniq
180109,180109,"Dizziness, hot flushes, nausea, pain in head around spine. Normal pulse","Hi, the symptoms left facial numbness, head ache, pain and constriction of the left hand and arm. The M.R.I. sh...","I just had an MRI which showed ""mild chronic small vessel ischemic changes in the periventricular and subcortical white matter of both c...",healthcaremagic


## 2. Normalise text
Both roles go through the same `clean_text()` as notebook 01:
- NFKC normalisation, which turns non-breaking spaces (common here) into plain spaces, `…` into `...`, and full-width commas into normal commas.
- ASCII quotes and dashes, and control characters removed.
- URLs, emails and phone numbers removed.
- A space added after a full stop where one is missing (`baby.i am` becomes `baby. i am`).
- Whitespace and punctuation spacing tidied.

Typos, hedging and informal style are kept.

In [4]:
CHAR_MAP = str.maketrans({"‘": "'", "’": "'", "´": "'", "`": "'", "“": '"', "”": '"',
                          "–": "-", "—": " - ", "•": "-", " ": " "})
URL_RE = re.compile(r"(?:https?:\\?/\\?/|www\.)\S+|\b[\w-]+\.(?:com|org|net|gov)\b(?:[/\\]\S*)?", re.I)
EMAIL_RE = re.compile(r"\S*@\S*")   # also catches masked "YYYY@YYYY" and leftovers like "@."
PHONE_RE = re.compile(r"(?:\+?\d{1,3}[-. ]?)?\(?\d{3}\)?[-. ]\d{3}[-. ]\d{4}\b")
MISSING_SPACE_RE = re.compile(r"(?<=[a-zA-Z]{3}[.?!,])(?=[a-zA-Z])")

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).translate(CHAR_MAP)
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C" or ch in "\n\t")
    text = EMAIL_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = PHONE_RE.sub(" ", text)
    text = MISSING_SPACE_RE.sub(" ", text)
    text = re.sub(r"\b(Hello|Hi|Dear|Thanks)(?=[A-Z])", r"\1 ", text)   # "HelloThanks" -> "Hello Thanks"
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = URL_RE.sub(" ", EMAIL_RE.sub(" ", text))   # second pass: "x@ yahoo .com" only joins up after the spacing fix
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([!?])\1{2,}", r"\1\1", text)
    text = re.sub(r"\.{4,}", "...", text)
    return text.strip(" -:").lstrip(".,;")

print(clean_text("hello doctor,i can not.i also did tests… mail me at a.b@x.com or 555-123-4567 https://www.icliniq.com/x"))

hello doctor, i can not. i also did tests... mail me at or


## 3. Patient-specific cleaning
- **Names:** self-introductions (`my name is samina`, `this is Mohit`) are removed, and a greeting to a named doctor (`hi dr. smith`) becomes `hi doctor`. The persona fields will supply identity later.
- **ALL-CAPS** messages are lowercased.
- **Optional `LOWERCASE`** makes the text match the COVID set's style.

In [5]:
NAME_INTRO_RE = re.compile(r"\bmy name is [a-z]+(?: [a-z]+)?\b[,.]?\s*", re.I)
THIS_IS_RE = re.compile(r"\bthis is [A-Z][a-z]+\b[,.]?\s*")   # "hi this is Mohit." (capitalised name only)
DR_GREETING_RE = re.compile(r"\b(hi|hello|hey|dear)\s+(?:dr\.?|doctor)\s+[A-Z][a-z]+\b", re.I)

def clean_patient(text: str) -> str:
    if FIX_ALL_CAPS and text.isupper():
        text = text.lower()
    text = NAME_INTRO_RE.sub("", text)
    text = THIS_IS_RE.sub("", text)
    text = DR_GREETING_RE.sub(lambda m: f"{m.group(1)} doctor", text)
    text = clean_text(text)
    return text.lower() if LOWERCASE else text

BOILERPLATE_RES = [re.compile(p, re.I) for p in DOCTOR_BOILERPLATE]

SITE_RE = re.compile(r"icliniq|health ?care ?magic|\bhcm\b|ask a doctor|chat ?doctor", re.I)
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

def clean_doctor(text: str) -> str:
    text = clean_text(text)
    # Any sentence that names the website is boilerplate, whatever its wording.
    text = " ".join(x for x in SENT_SPLIT_RE.split(text) if not SITE_RE.search(x))
    for _ in range(4):
        before = text
        for rx in BOILERPLATE_RES:
            text = rx.sub("", text).strip(" ,.-:>")
        if text == before:
            break
    text = text.lower() if LOWERCASE else text
    return text if len(text.split()) >= MIN_DOCTOR_WORDS else ""

df["patient"] = df["Patient"].map(clean_patient)
df["doctor"] = df["Doctor"].map(clean_doctor)
df["description"] = df["Description"].map(clean_text)

for _, r in df.sample(3, random_state=7).iterrows():
    print("PATIENT RAW  :", r.Patient[:250]); print("PATIENT CLEAN:", r.patient[:250])
    print("DOCTOR RAW   :", r.Doctor[:250]);  print("DOCTOR CLEAN :", r.doctor[:250]); print("-" * 100)

PATIENT RAW  : Hello doctor, I have severe anxiety problems which cripple me. I feel trouble in concentrating on my studies and other daily activities. I have PCOS and suffering from depression. I am an introvert, and my body is swaying all the time. A feeling of c
PATIENT CLEAN: Hello doctor, I have severe anxiety problems which cripple me. I feel trouble in concentrating on my studies and other daily activities. I have PCOS and suffering from depression. I am an introvert, and my body is swaying all the time. A feeling of c
DOCTOR RAW   : Hello. You should work on your weight loss. It will help you with both your PCOS and depression. Do the daily exercise for 30 to 45 minutes. Do meditation for 20 minutes daily. Avoid junk foods, eat healthy joint cooked foods. Take enough 6 to 7 hour
DOCTOR CLEAN : Hello. You should work on your weight loss. It will help you with both your PCOS and depression. Do the daily exercise for 30 to 45 minutes. Do meditation for 20 minutes daily. Avoid junk

## 4. Filter
Rows are dropped only for **patient-side** problems, because the patient text is what gets trained. A doctor reply that is empty after cleaning is blanked and flagged (`doctor_usable = False`) instead. It is not in the training string anyway, and the flag lets a future doctor-model pipeline skip those rows.

Each drop is recorded with a reason.

In [6]:
EN_FUNCTION_WORDS = set("i me my the a an and or but is am are was were be been have has had do does did "
                        "to of in on for with at from it this that what how can should will would not no "
                        "you your he she his her they we".split())
FOREIGN_FUNCTION_WORDS = set("ek het die nie ons maar ook vir sal wat jy hulle el la que los las tengo una "
                             "por con del es muy pero je suis est les des une pas ich und ist nicht".split())
ATTACH_RE = re.compile(r"\battach(?:ed|ing|ment)?\b|\bfind (?:my|the) reports?\b|\bsee (?:the )?(?:attached|pics?|photos?)\b", re.I)

def drop_reason(text: str):
    toks = text.split()
    if len(toks) < MIN_WORDS:                                  return f"patient < {MIN_WORDS} words"
    if len(toks) > MAX_WORDS:                                  return f"patient > {MAX_WORDS} words"
    words = [w.lower().strip(".,!?()") for w in toks]
    if sum(w in FOREIGN_FUNCTION_WORDS for w in words) > sum(w in EN_FUNCTION_WORDS for w in words):
        return "likely non-English"
    if sum(ord(c) > 127 for c in text) > 0.2 * len(text):      return "mostly non-ASCII"
    if sum(any(c.isdigit() for c in t) for t in toks) / len(toks) > MAX_DIGIT_RATIO:
        return "lab-report dump"
    if DROP_ATTACHMENT_REFS and ATTACH_RE.search(text):        return "references attachment"
    return None

df["drop_reason"] = df["patient"].map(drop_reason)
df["doctor_usable"] = df["doctor"] != ""

## 5. Deduplicate, decontaminate, split
- **Dedup key:** the patient text, lowercased, with punctuation and spacing removed. The first occurrence is kept (about 11k exact duplicates in the raw data).
- **Decontamination:** removes any row whose key matches a dev or test patient message from notebook 01's COVID set, so the COVID evaluation stays clean if the two datasets are combined.
- **Split:** `md5(key) % 100` assigns train, dev or test. It is deterministic, doesn't depend on `MAX_ROWS` order, and duplicates can never end up in two different splits.

In [7]:
def norm_key(text: str) -> str:
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", text.lower()).split())

df["key"] = df["patient"].map(norm_key)
dup = df["drop_reason"].isna() & df.duplicated("key", keep="first")
df.loc[dup, "drop_reason"] = "duplicate"

excluded = set()
for path in EXCLUDE_JSONL:
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            d = json.loads(line)
            excluded.update(norm_key(t["text"]) for t in d["turns"] if t["role"] == TRAIN_ROLE)
leak = df["drop_reason"].isna() & df["key"].isin(excluded)
df.loc[leak, "drop_reason"] = "overlaps COVID dev/test"
print(f"loaded {len(excluded)} held-out keys; removed {leak.sum()} overlaps")

def assign_split(key: str) -> str:
    b = int(hashlib.md5(key.encode()).hexdigest(), 16) % 100
    edge = 0
    for s, pct in SPLIT_PCT.items():
        edge += pct
        if b < edge:
            return s
df["split"] = df["key"].map(assign_split)

kept = df[df["drop_reason"].isna()].copy()
report = (df.assign(status=df["drop_reason"].fillna("kept"))
            .pivot_table(index="status", columns="source", values="id", aggfunc="count", fill_value=0))
report.loc["TOTAL"] = report.sum()
report

loaded 121 held-out keys; removed 31 overlaps


source,healthcaremagic,icliniq
status,,
duplicate,658,9434
kept,222894,18488
lab-report dump,111,4
mostly non-ASCII,2,0
overlaps COVID dev/test,29,2
patient < 8 words,202,10
patient > 300 words,1797,668
references attachment,712,2458
TOTAL,226405,31064


In [8]:
pd.crosstab(kept["split"], kept["source"], margins=True)

source,healthcaremagic,icliniq,All
split,,,
dev,11054,948,12002
test,11196,970,12166
train,200644,16570,217214
All,222894,18488,241382


In [9]:
# A few examples of each drop reason, to sanity-check the filters.
for reason, g in df[df["drop_reason"].notna()].groupby("drop_reason"):
    print(f"== {reason} ({len(g):,})")
    for t in g["patient"].head(2):
        print("   ", t[:160])

== duplicate (10,092)
    Hi doctor, I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bul
    Hello doctor, I am 48 years old. I am experiencing weak erection and difficulty in sustaining the same. This condition was observed 10 years back. Also, there i
== lab-report dump (115)
    Hello doctor, Should I think about going to a hematologist? The last few years of my CBC shows WBC 8.4, 9.6, 13.9, 11.3, 11.3, 10.5, 8.3, 12.3, lymph (abs) 3.2,
    Hello doctor, What does this vision screening mean? OD +0.75 +1.75 -200 156 degrees. OS +2.75 +3.50 -100 4 degrees.
== mostly non-ASCII (2)
    мy ғιance нaѕ вeen developιng lυмpѕ all oғ нιѕ вacĸ and тнey pop υp oυт oғ ĸnow wнere. нe doeѕ noт нave ι ѕυrance yeт ѕo we can'т тaĸe нιм тo тнe docтor вυт тнe
    सर,अगर टेस्टेस मे दर्द ना होकर सिर्फ़ सूजन हो तो क्या इसे भी टेस्टिक्युलर टॉर्जन बीमारी कहना चाहिए... कृपया प्रति उत्तर दे...
== overlaps COVID d

## 6. Before / after report

In [10]:
def wstats(s):
    return s.str.split().str.len().describe(percentiles=[.05, .5, .95, .99]).round(1)[["count", "mean", "5%", "50%", "95%", "99%", "max"]]

pd.DataFrame({"patient raw": wstats(df["Patient"]), "patient clean (kept)": wstats(kept["patient"]),
              "doctor raw": wstats(df["Doctor"]), "doctor clean (kept)": wstats(kept["doctor"])}).T

,count,mean,5%,50%,95%,99%,max
patient raw,257469.0,83.6,42.0,69.0,171.0,296.0,3672.0
patient clean (kept),241382.0,80.4,44.0,68.0,162.0,237.0,300.0
doctor raw,257469.0,89.3,24.0,79.0,188.0,298.0,1867.0
doctor clean (kept),241382.0,79.0,19.0,69.0,172.0,256.0,1898.0


In [11]:
print("doctor_usable share among kept:", kept["doctor_usable"].mean().round(3))
print(kept.groupby("source")["doctor_usable"].mean().round(3).to_string())

# Sanity checks: no leftover URLs/emails/boilerplate in the kept text.
for col in ("patient", "doctor"):
    assert not kept[col].str.contains(URL_RE).any(), f"URL left in {col}"
    assert not kept[col].str.contains(EMAIL_RE).any(), f"email left in {col}"
left = kept["doctor"].str.contains(SITE_RE) | kept["doctor"].str.contains("attachment removed", case=False)
print("doctor boilerplate left:", int(left.sum()))
assert left.mean() < 0.005, kept.loc[left, "doctor"].head(3).tolist()
print("clean OK")

doctor_usable share among kept: 0.981
source
healthcaremagic    0.990
icliniq            0.877


doctor boilerplate left: 28
clean OK


## 7. Save the cleaned dialogues
The schema matches notebook 01, so the two can be concatenated later.

In [12]:
clean_dir = OUT_DIR / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)
for s in SPLITS:
    with open(clean_dir / f"{s}.jsonl", "w", encoding="utf-8") as f:
        for r in kept[kept["split"] == s].itertuples():
            turns = [{"role": "patient", "text": r.patient}] + ([{"role": "doctor", "text": r.doctor}] if r.doctor_usable else [])
            f.write(json.dumps({"dialogue_id": f"dm-{r.id:06d}", "split": s, "source": r.source,
                                "description": r.description, "doctor_usable": bool(r.doctor_usable),
                                "turns": turns}, ensure_ascii=False) + "\n")
df.loc[df["drop_reason"].notna(), ["id", "source", "drop_reason", "Patient"]].to_csv(clean_dir / "dropped.csv", index=False)
print("saved ->", clean_dir, {s: int((kept["split"] == s).sum()) for s in SPLITS})

saved -> C:\Users\Marcus\Documents\GitHub\cs425\model\data\processed\diagnose-me\clean {'train': 217214, 'dev': 12002, 'test': 12166}


## 8. Format, tokenize and mask
This uses the same string format and masking rule as notebook 01:
```
<|doctor|>What brings you in today?<|patient|>Hi doctor, I have had ... <|doctor|>
└──────────── masked ───────────────┘└marker┘└──── loss computed here ────┘└ loss ┘
```
`labels` is `-100` on every token that does not start inside the patient span. Tokenization is batched through `Dataset.map` with a fast tokenizer's offset mapping.

In [13]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "a fast tokenizer is needed for offset mapping"
base_vocab = len(tokenizer)
tokenizer.add_special_tokens({"additional_special_tokens": list(ROLE_TOKENS.values()) + EXTRA_SPECIAL_TOKENS})
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
MAX_LEN = min(MAX_LENGTH, tokenizer.model_max_length)
MODEL_SLUG = MODEL_NAME.replace("/", "__")
print(f"{MODEL_NAME}: vocab {base_vocab} -> {len(tokenizer)}; eos={tokenizer.eos_token!r} pad={tokenizer.pad_token!r}; max_len={MAX_LEN}")

def format_row(patient: str, doctor: str):
    """Return (text, [start, end)) of the trained span."""
    text = (ROLE_TOKENS["doctor"] + DOCTOR_OPENER) if ADD_DOCTOR_OPENER else ""
    text += ROLE_TOKENS["patient"]
    start = len(text)
    text += patient + ROLE_TOKENS["doctor"]           # the next role marker is the stop signal
    end = len(text)
    if not TRIM_AFTER_LAST_TARGET and doctor:
        text += doctor + tokenizer.eos_token
    return text, start, end

def encode(batch):
    texts, starts, ends = zip(*(format_row(p, d) for p, d in zip(batch["patient"], batch["doctor"])))
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    labels, n_train = [], []
    for ids, offs, a, b in zip(enc["input_ids"], enc.pop("offset_mapping"), starts, ends):
        lab = [t if (e > s and a <= s < b) else -100 for t, (s, e) in zip(ids, offs)]
        labels.append(lab); n_train.append(sum(l != -100 for l in lab))
    enc["labels"], enc["n_train_tokens"] = labels, n_train
    enc["truncated"] = [len(ids) == MAX_LEN for ids in enc["input_ids"]]
    return enc

dsd = DatasetDict({
    s: Dataset.from_pandas(kept.loc[kept["split"] == s, ["id", "patient", "doctor"]].reset_index(drop=True))
    for s in SPLITS
})
dsd = dsd.map(encode, batched=True, batch_size=1000, remove_columns=["patient", "doctor"], desc="tokenizing")
dsd = dsd.filter(lambda x: x["n_train_tokens"] > 0)

pd.DataFrame({s: {
    "examples": len(dsd[s]),
    "truncated": int(sum(dsd[s]["truncated"])),
    "tokens p50": int(pd.Series([len(x) for x in dsd[s]["input_ids"]]).median()),
    "tokens p95": int(pd.Series([len(x) for x in dsd[s]["input_ids"]]).quantile(.95)),
    "trainable tokens": int(sum(dsd[s]["n_train_tokens"])),
} for s in SPLITS}).T

gpt2: vocab 50257 -> 50263; eos='<|endoftext|>' pad='<|endoftext|>'; max_len=512


tokenizing:   0%|          | 0/217214 [00:00<?, ? examples/s]

tokenizing:   0%|          | 0/12002 [00:00<?, ? examples/s]

tokenizing:   0%|          | 0/12166 [00:00<?, ? examples/s]

Filter:   0%|          | 0/217214 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12002 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12166 [00:00<?, ? examples/s]

,examples,truncated,tokens p50,tokens p95,trainable tokens
train,217214,18,92,208,21672901
dev,12002,2,93,205,1191567
test,12166,1,93,211,1219482


In [14]:
def show_mask(ex, limit=100):
    """Trained tokens in «», masked tokens plain."""
    print("".join(f"«{tokenizer.decode([t])}»" if l != -100 else tokenizer.decode([t])
                  for t, l in list(zip(ex["input_ids"], ex["labels"]))[:limit]))

show_mask(dsd["train"][0])

<|doctor|>What brings you in today?<|patient|>«Hi»« doctor»«,»« I»« am»« just»« wondering»« what»« is»« ab»«ut»«ting»« and»« ab»«ut»«ment»« of»« the»« nerve»« root»« means»« in»« a»« back»« issue»«.»« Please»« explain»«.»« What»« treatment»« is»« required»« for»« ann»«ular»« bul»«ging»« and»« tear»«?»«<|doctor|>»


In [15]:
# Check on a sample: the trained tokens decode to exactly the patient text plus the stop marker.
import random
by_id = dict(zip(kept["id"], kept["patient"]))
rng = random.Random(SEED)
for s in SPLITS:
    for i in rng.sample(range(len(dsd[s])), min(500, len(dsd[s]))):
        ex = dsd[s][i]
        assert len(ex["input_ids"]) <= MAX_LEN
        trained = tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])
        body = trained.replace(ROLE_TOKENS["doctor"], "").strip()
        if not ex["truncated"]:
            assert trained.endswith(ROLE_TOKENS["doctor"]), trained[-50:]
        assert by_id[ex["id"]].startswith(body[:200].strip()) or body in by_id[ex["id"]], (ex["id"], body[:100])
print("mask OK")

mask OK


## 9. Save the tokenized dataset

In [16]:
tok_dir = OUT_DIR / "tokenized" / MODEL_SLUG
dsd.remove_columns(["n_train_tokens", "truncated"]).save_to_disk(str(tok_dir / "dataset"))
tokenizer.save_pretrained(str(tok_dir / "tokenizer"))

prep_config = {
    "source_file": str(RAW_FILE.relative_to(ROOT)), "model_name": MODEL_NAME, "max_length": MAX_LEN,
    "max_rows": MAX_ROWS, "train_role": TRAIN_ROLE, "add_doctor_opener": ADD_DOCTOR_OPENER,
    "doctor_opener": DOCTOR_OPENER, "trim_after_last_target": TRIM_AFTER_LAST_TARGET,
    "role_tokens": ROLE_TOKENS, "extra_special_tokens": EXTRA_SPECIAL_TOKENS, "vocab_size": len(tokenizer),
    "stop_token": ROLE_TOKENS["doctor"], "lowercase": LOWERCASE, "fix_all_caps": FIX_ALL_CAPS,
    "min_words": MIN_WORDS, "max_words": MAX_WORDS, "max_digit_ratio": MAX_DIGIT_RATIO,
    "drop_attachment_refs": DROP_ATTACHMENT_REFS, "split_pct": SPLIT_PCT, "seed": SEED,
    "counts": {s: len(dsd[s]) for s in SPLITS},
    "dropped": df["drop_reason"].value_counts().to_dict(),
}
(tok_dir / "prep_config.json").write_text(json.dumps(prep_config, indent=2), encoding="utf-8")
print("saved ->", tok_dir)
dsd

Saving the dataset (0/1 shards):   0%|          | 0/217214 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12002 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12166 [00:00<?, ? examples/s]

saved -> C:\Users\Marcus\Documents\GitHub\cs425\model\data\processed\diagnose-me\tokenized\gpt2


DatasetDict({
    train: Dataset({
        features: ['id', 'input_ids', 'attention_mask', 'labels', 'n_train_tokens', 'truncated'],
        num_rows: 217214
    })
    dev: Dataset({
        features: ['id', 'input_ids', 'attention_mask', 'labels', 'n_train_tokens', 'truncated'],
        num_rows: 12002
    })
    test: Dataset({
        features: ['id', 'input_ids', 'attention_mask', 'labels', 'n_train_tokens', 'truncated'],
        num_rows: 12166
    })
})

## 10. Notes for training
- Load the data exactly as with notebook 01: `load_from_disk(f"{tok_dir}/dataset").remove_columns("id")`, then `model.resize_token_embeddings(len(tokenizer))` and `DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)`.
- This dataset teaches **how real patients write** an opening complaint. It is single-turn, so it cannot teach multi-turn consistency. Use it as the first training round, then fine-tune on the synthetic case-card dialogues.
- To mix it with the COVID set, set `LOWERCASE = True` here, or turn it off in notebook 01, so the casing matches. Both write the same clean JSONL schema.